<a href="https://colab.research.google.com/github/athfizh/PemMes26_05_Hafizh/blob/main/JS09/JS09-03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **Approximate Nearest Neighbors (ANN) - Lab 3**

---

> Lab 3 masih membandingkan exact NN dengan metode ANN, tapi kali ini lawannya **HNSW** (Hierarchical Navigable Small World). Datanya 1.000 titik 2D, query di <code>(0.5, 0.5)</code>, dan kita cari 5 tetangga terdekat. Perhatikan bahwa HNSW memakai struktur graf bertingkat, bukan pohon seperti ANNOY.

---

# **Langkah 1 - Install Library**

---

In [1]:
!pip install -q hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


---

# **Langkah 2 - Import Library**

---

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

> <code>hnswlib</code> adalah implementasi HNSW yang ringan dan cepat. <code>NearestNeighbors</code> dari scikit-learn dipakai sebagai pembanding exact NN dengan metode brute force.

---

# **Langkah 3 - Buat Dataset Dummy**

---

In [3]:
# Build Dummy Dataset
np.random.seed(42)  # biar hasil bisa diulang
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

print("Shape data:", data.shape)
print("Query:", query)

Shape data: (1000, 2)
Query: [[0.5 0.5]]


---

# **Langkah 4 - Buat Model NN**

---

In [4]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performance
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "s")

=== Exact NN ===
Indices: [[137 170 750 316 991]]
Distances: [[0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]]
Waktu: 0.05679202079772949 s


> <code>algorithm='brute'</code> memaksa scikit-learn menghitung jarak ke semua titik, jadi ini benar-benar exact NN. <code>metric='euclidean'</code> memakai jarak L2. Waktu yang diukur adalah waktu pencarian, bukan waktu training, karena brute force tidak benar-benar belajar apa pun.

---

# **Langkah 5 - Buat Model HNSW**

---

In [5]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "s")


=== HNSW ===
Indices: [[137 170 750 316 991]]
Distances: [[0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]]
Waktu: 0.0002837181091308594 s


> <code>space='l2'</code> berarti HNSW memakai jarak Euclidean. <code>M=16</code> menentukan jumlah koneksi tiap node di graf, dan <code>ef_construction=100</code> menentukan seberapa teliti graf dibangun. Saat query, <code>set_ef(50)</code> mengatur seberapa banyak kandidat yang diperiksa: semakin besar <code>ef</code>, semakin akurat tapi semakin lambat.

> HNSW bekerja dengan **greedy search** di graf bertingkat. Dari level atas yang jarang, pencarian turun ke level bawah yang lebih rapat sampai tidak ada tetangga yang lebih dekat. Karena hanya menjelajahi sebagian kecil node, HNSW bisa jauh lebih cepat daripada brute force.